# Lab Day 10: Finetune Qwen3-VL-4B (QLoRA) để định vị hư hại mặt đường
Bản chạy lại từ đầu, tối ưu cho Colab T4. Khác bản cũ:
- Prompt ngắn (tiếng Anh) → ít token hơn mỗi mẫu → mỗi bước train nhanh hơn.
- Tắt gradient checkpointing mặc định (tự bật lại nếu OOM, và ghi vào `run.json`).
- Chuẩn bị batch ở luồng nền, đo thời gian chờ dữ liệu.
- Sửa lỗi `grad_dtype` của projector ở E2/E3.
- E4 chọn biến `data_size` (10% vs 100%), 100% dùng lại kết quả setup tốt nhất.
- Tự sao lưu adapter + evidence lên Google Drive.

## 0. Môi trường (Colab T4)

In [ ]:
!git clone https://github.com/dangkhoabui15092004-byte/K4-Track-4-Day10-VisionLanguageModel.git
%cd K4-Track-4-Day10-VisionLanguageModel

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
# Sao lưu lên Google Drive (session Colab free hay bị ngắt). Không mount được thì bỏ qua.
import shutil
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/lab_day10_backup")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE.mkdir(parents=True, exist_ok=True)
except Exception as e:
    DRIVE = None
    print("Không dùng được Drive:", e)


def backup(setup=None, save_dir="work/adapters"):
    """Chép adapter bước cuối + run.json (nếu có setup) và toàn bộ evidence lên Drive."""
    if DRIVE is None:
        return
    if setup:
        src = Path(save_dir) / setup
        dst = DRIVE / Path(save_dir).name / setup
        dst.mkdir(parents=True, exist_ok=True)
        if (src / "run.json").exists():
            shutil.copy2(src / "run.json", dst / "run.json")
        for step_dir in sorted(src.glob("step*"), key=lambda p: int(p.name[4:])):
            last = step_dir
        shutil.copytree(last, dst / last.name, dirs_exist_ok=True)
    ev = Path("report/evidence")
    if ev.exists():
        shutil.copytree(ev, DRIVE / "evidence", dirs_exist_ok=True)


def restore():
    """Khôi phục từ Drive sau khi session bị ngắt."""
    if DRIVE is None or not DRIVE.exists():
        return
    for d in DRIVE.iterdir():
        if d.name == "evidence":
            shutil.copytree(d, "report/evidence", dirs_exist_ok=True)
        else:
            shutil.copytree(d, Path("work") / d.name, dirs_exist_ok=True)
    print("đã khôi phục từ", DRIVE)


# Nếu vừa bị ngắt session và Drive đã có bản sao lưu, bỏ comment dòng dưới (SAU KHI chạy mục 1 để có thư mục work/):
# restore()

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json, random
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
for s, rows in splits.items():
    n_boxes = sum(len(r["objects"]) for r in rows)
    print(f"{s:5s} {len(rows):5d} ảnh  {n_boxes:5d} box")

# không session nào xuất hiện ở hai split
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
overlap = (sess["train"] & sess["val"]) | (sess["train"] & sess["test"]) | (sess["val"] & sess["test"])
print("session trùng giữa các split:", overlap if overlap else "không có (đúng như kỳ vọng)")

# vẽ 6 ảnh có box để chắc hiểu đúng nhãn YOLO (xc, yc, w, h chuẩn hoá [0,1])
random.seed(0)
sample = random.sample([r for r in splits["train"] if r["objects"]], 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, r in zip(axes.flat, sample):
    img = Image.open(f"data/{r['image_path']}").convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = r["width"], r["height"]
    for o in r["objects"]:
        xc, yc, bw, bh = o["yolo"]
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1 - 12)), o["label"], fill="red")
    ax.imshow(img)
    ax.set_title(f"{r['image_id']}  ({r['session']})", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

CLASSES = ["pothole", "crack", "manhole"]


def yolo_to_box(label, yolo):
    """YOLO [xc, yc, w, h] chuẩn hoá [0,1] -> {"label", "bbox_2d": [x1,y1,x2,y2]} thang 0-1000."""
    xc, yc, w, h = yolo
    x1, y1 = (xc - w / 2) * 1000, (yc - h / 2) * 1000
    x2, y2 = (xc + w / 2) * 1000, (yc + h / 2) * 1000

    def clamp(v):
        return max(0, min(1000, round(v)))

    return {"label": label, "bbox_2d": [clamp(x1), clamp(y1), clamp(x2), clamp(y2)]}


def objects_to_answer(objects):
    """list[{"label","yolo"}] -> chuỗi JSON đáp án, sắp theo (y1, x1)."""
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))


# Prompt NGẮN (tiếng Anh): ít token hơn nhiều so với bản tiếng Việt → train/suy luận nhanh hơn.
# Dùng CHUNG một prompt cho E0–E4.
PROMPT = (
    "Detect road damage in this dashcam image. Classes: pothole, crack, manhole.\n"
    "Reply with ONLY a JSON list, no other text, no code fences. "
    "Each item: {\"label\": <class>, \"bbox_2d\": [x1, y1, x2, y2]}.\n"
    "Coordinates are integers on a 0-1000 scale relative to image width/height (NOT pixels). "
    "If nothing is present, reply []."
)
print(PROMPT)

In [ ]:
# Kiểm tra: nhãn -> JSON -> quy ngược về pixel -> vẽ lại, phải trùng box gốc
sample = next(r for r in splits["train"] if r["objects"])
answer = objects_to_answer(sample["objects"])
print("JSON đáp án:", answer)

boxes_back = json.loads(answer)
img = Image.open(f"data/{sample['image_path']}").convert("RGB")
draw = ImageDraw.Draw(img)
w, h = sample["width"], sample["height"]
for b in boxes_back:
    x1, y1, x2, y2 = b["bbox_2d"]
    draw.rectangle([x1 / 1000 * w, y1 / 1000 * h, x2 / 1000 * w, y2 / 1000 * h], outline="lime", width=3)
plt.imshow(img)
plt.axis("off")
plt.title("quy ngược từ JSON 0-1000 về pixel")
plt.show()

## 3. Khai báo model + E0 zero-shot

In [ ]:
import gc, time, subprocess
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"


def build_quantization_config():
    # ViT + lm_head KHÔNG lượng tử hoá: nếu không, merger/projector ở E2-E3 không huấn luyện đầy đủ được.
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"],
    )


def fresh_base_model():
    return Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=build_quantization_config(), dtype=torch.float16, device_map={"": 0},
    )


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"


def run_inference(model, processor, entries, out_path, prompt=PROMPT, batch_size=8, max_new_tokens=192):
    """Suy luận trên `entries`, ghi work/*.jsonl: {"image_id","raw"} mỗi dòng."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    with out_path.open("w", encoding="utf-8") as f:
        for i in range(0, len(entries), batch_size):
            batch = entries[i:i + batch_size]
            images = [Image.open(f"data/{e['image_path']}").convert("RGB") for e in batch]
            messages_batch = [
                [{"role": "user", "content": [{"type": "image", "image": img},
                                               {"type": "text", "text": prompt}]}]
                for img in images
            ]
            inputs = processor.apply_chat_template(
                messages_batch, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True,
            ).to(model.device)
            prompt_len = inputs["input_ids"].shape[1]
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            raws = processor.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
            for e, raw in zip(batch, raws):
                f.write(json.dumps({"image_id": e["image_id"], "raw": raw}, ensure_ascii=False) + "\n")
            print(f"  {min(i + batch_size, len(entries))}/{len(entries)}", end="\r")
    print()

In [ ]:
# E0: zero-shot trên test (bỏ qua nếu đã có kết quả)
if Path("report/evidence/e0.json").exists():
    print("E0 đã có kết quả, bỏ qua")
else:
    model = fresh_base_model()
    t0 = time.time()
    run_inference(model, processor, splits["test"], "work/e0.jsonl")
    print(f"E0 suy luận xong trong {(time.time() - t0) / 60:.1f} phút")
    del model
    free_gpu()
    !python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
    backup()

In [ ]:
# Kiểm tra: parse_valid_rate và 5 output thô (có bọc ```json không? có chữ thừa không?)
import sys
sys.path.insert(0, ".")
from scripts.score import parse_output

for row in read_jsonl("work/e0.jsonl")[:5]:
    print(row["image_id"], "->", repr(row["raw"])[:200])
    print("   parse_output:", parse_output(row["raw"]))

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
IGNORE_INDEX = -100


def encode_example(image, prompt, answer, processor):
    """Trả dict đã tokenize; labels = -100 trên prompt + token ảnh, chỉ giữ loss trên JSON trả lời."""
    user_msg = [{"role": "user", "content": [{"type": "image", "image": image},
                                               {"type": "text", "text": prompt}]}]
    assistant_msg = user_msg + [{"role": "assistant", "content": answer}]

    prompt_enc = processor.apply_chat_template(
        user_msg, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    full_enc = processor.apply_chat_template(
        assistant_msg, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")

    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), (
        "prompt không phải tiền tố của chuỗi đầy đủ — kiểm tra lại chat template")

    labels = full_enc["input_ids"].clone()
    labels[:, :n] = IGNORE_INDEX
    full_enc["labels"] = labels
    return full_enc


def collate(batch):
    """Pad input_ids/attention_mask/labels (và mm_token_type_ids nếu có); pixel_values, image_grid_thw: cat dim 0."""
    pad_id = processor.tokenizer.pad_token_id
    text_keys = [k for k in ("input_ids", "attention_mask", "labels", "mm_token_type_ids") if k in batch[0]]
    max_len = max(b["input_ids"].shape[1] for b in batch)

    out = {}
    for k in text_keys:
        pad_value = IGNORE_INDEX if k == "labels" else (pad_id if k == "input_ids" else 0)
        rows = []
        for b in batch:
            t = b[k][0]
            pad_len = max_len - t.shape[0]
            if pad_len:
                t = torch.cat([t, torch.full((pad_len,), pad_value, dtype=t.dtype)])
            rows.append(t)
        out[k] = torch.stack(rows)

    out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch], dim=0)
    out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    return out

In [ ]:
# Kiểm tra: số token ảnh phải là 220; phần labels != -100 phải đúng là JSON đáp án
ex = next(r for r in splits["train"] if r["objects"])
img = Image.open(f"data/{ex['image_path']}").convert("RGB")
answer = objects_to_answer(ex["objects"])
enc = encode_example(img, PROMPT, answer, processor)

image_pad_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
n_image_tokens = int((enc["input_ids"][0] == image_pad_id).sum())
print("số token ảnh:", n_image_tokens, "(kỳ vọng 220)")
print("tổng số token của mẫu:", enc["input_ids"].shape[1])

mask = enc["labels"][0] != IGNORE_INDEX
print("phần được tính loss (phải là JSON đáp án + <|im_end|>):")
print(processor.tokenizer.decode(enc["input_ids"][0][mask]))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
from peft import LoraConfig, get_peft_model

LLM_PATTERN = r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
VIT_PATTERN = r"model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))"
PROJ_KEYS = ("visual.merger", "visual.deepstack_merger_list")


def apply_setup(model, setup, r=16, lora_alpha=None):
    """E1: LoRA LLM. E2: + projector đầy đủ (fp32). E3: + LoRA ViT."""
    assert setup in ("E1", "E2", "E3")
    lora_alpha = lora_alpha or 2 * r

    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()

    pattern = LLM_PATTERN if setup != "E3" else f"(?:{LLM_PATTERN})|(?:{VIT_PATTERN})"
    matched = [n for n, _ in model.named_modules() if re.fullmatch(pattern, n)]
    n_llm = model.config.text_config.num_hidden_layers * 7
    n_vit = model.config.vision_config.depth * 4
    expected = n_llm if setup in ("E1", "E2") else n_llm + n_vit
    print(f"{setup}: {len(matched)} module khớp LoRA (kỳ vọng {expected}; 4B = 252/348)")
    assert len(matched) == expected, "số module LoRA không khớp — in named_modules() để kiểm tra regex"

    peft_model = get_peft_model(model, LoraConfig(
        r=r, lora_alpha=lora_alpha, lora_dropout=0.05, target_modules=matched, task_type="CAUSAL_LM"))

    if setup in ("E2", "E3"):
        # THAY HẲN Parameter bằng bản fp32 mới (không dùng p.data = ...): cách cũ để lại
        # grad_dtype = fp16 trên tensor → lỗi "attempting to assign a gradient with dtype 'float'".
        for n, p in list(peft_model.named_parameters()):
            if any(k in n for k in PROJ_KEYS):
                mod_name, _, pname = n.rpartition(".")
                mod = peft_model.get_submodule(mod_name)
                setattr(mod, pname, torch.nn.Parameter(p.detach().float().contiguous(), requires_grad=True))

    n_trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"{setup}: tổng tham số huấn luyện = {n_trainable:,}")
    return peft_model, matched

## 6. Vòng huấn luyện

In [ ]:
import json
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from torch.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup


def sample_batch(entries, batch_size, processor):
    picks = random.sample(entries, batch_size) if len(entries) >= batch_size else random.choices(entries, k=batch_size)
    encs = []
    for e in picks:
        img = Image.open(f"data/{e['image_path']}").convert("RGB")
        encs.append(encode_example(img, PROMPT, objects_to_answer(e["objects"]), processor))
    return collate(encs)


def train_setup(model, processor, setup, train_entries, max_steps=300, lr=2e-4,
                batch_size=2, grad_accum=4, save_every=100, save_dir="work/adapters", r=16,
                grad_ckpt=False):
    peft_model, _ = apply_setup(model, setup, r=r)
    out_dir = Path(save_dir) / setup
    out_dir.mkdir(parents=True, exist_ok=True)

    trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
    print(f"{setup}: dtype các tham số huấn luyện =", {str(p.dtype) for p in trainable_params})
    assert all(p.dtype == torch.float32 for p in trainable_params), "có tham số huấn luyện không phải fp32"
    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * max_steps)), max_steps)
    scaler = GradScaler("cuda")

    if grad_ckpt:
        peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_model.config.use_cache = False
    peft_model.train()
    print(f"{setup}: gradient checkpointing = {grad_ckpt}")

    # Chuẩn bị batch ở 1 luồng nền (đúng 1 luồng: tokenizer nhanh của HF không an toàn khi dùng song song)
    pool = ThreadPoolExecutor(max_workers=1)
    queue = deque(pool.submit(sample_batch, train_entries, batch_size, processor) for _ in range(2))

    def save_checkpoint(step):
        ckpt_dir = out_dir / f"step{step}"
        peft_model.save_pretrained(str(ckpt_dir))
        if setup in ("E2", "E3"):
            # peft chỉ lưu LoRA: projector huấn luyện đầy đủ phải lưu riêng
            extra = {n: p.detach().cpu() for n, p in peft_model.named_parameters()
                     if any(k in n for k in PROJ_KEYS) and p.requires_grad}
            torch.save(extra, ckpt_dir / "extra_trainable.pt")

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    data_wait = 0.0
    last_loss = None
    try:
        for step in range(1, max_steps + 1):
            opt.zero_grad()
            for _ in range(grad_accum):
                fut = queue.popleft()
                queue.append(pool.submit(sample_batch, train_entries, batch_size, processor))
                tw = time.time()
                cpu_batch = fut.result()
                data_wait += time.time() - tw
                batch = {k: v.to(peft_model.device) for k, v in cpu_batch.items()}
                with autocast(device_type="cuda", dtype=torch.float16):
                    loss = peft_model(**batch).loss / grad_accum
                if not torch.isfinite(loss):
                    raise RuntimeError(f"loss không hữu hạn ở bước {step}: {loss.item()} (nguy cơ tràn số fp16)")
                scaler.scale(loss).backward()
                last_loss = loss.item() * grad_accum

            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()

            if step % 20 == 0 or step == max_steps:
                el = time.time() - t0
                print(f"[{setup}] bước {step}/{max_steps}  loss={last_loss:.4f}  "
                      f"{el / step:.1f}s/bước  chờ dữ liệu {data_wait / 60:.1f} phút")
            if step % save_every == 0 or step == max_steps:
                save_checkpoint(step)
    finally:
        pool.shutdown(wait=False, cancel_futures=True)

    run_info = {
        "setup": setup, "rank": r,
        "trainable_params": sum(p.numel() for p in trainable_params),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "train_minutes": round((time.time() - t0) / 60, 2),
        "final_step": max_steps,
        "grad_checkpointing": grad_ckpt,
        "data_wait_minutes": round(data_wait / 60, 2),
        "n_train_images": len(train_entries),
    }
    (out_dir / "run.json").write_text(json.dumps(run_info, indent=2, ensure_ascii=False), encoding="utf-8")
    print(run_info)
    return peft_model, run_info


def train_with_fallback(setup, train_entries, save_dir="work/adapters", r=16, **kw):
    """Thử không gradient checkpointing trước (nhanh hơn); nếu OOM thì tự bật lại và train lại."""
    for ckpt in (False, True):
        base = fresh_base_model()
        pm, info, oom = None, None, False
        try:
            pm, info = train_setup(base, processor, setup, train_entries,
                                   save_dir=save_dir, r=r, grad_ckpt=ckpt, **kw)
        except torch.cuda.OutOfMemoryError:
            oom = True
        pm = base = None          # bỏ tham chiếu (ngoài khối except) để giải phóng VRAM
        free_gpu()
        if not oom:
            return info
        print(f"!!! {setup}: OOM khi grad_ckpt={ckpt}" + (" → thử lại với gradient checkpointing" if not ckpt else ""))
    raise RuntimeError(f"{setup}: vẫn OOM dù đã bật gradient checkpointing; giảm batch_size=1, grad_accum=8")

In [ ]:
# Huấn luyện E1, E2, E3. Setup đã có run.json thì BỎ QUA (chạy lại được sau khi session bị ngắt).
# Muốn train lại một setup: xoá thư mục work/adapters/<setup>.
for name in ("model", "base", "peft_model", "infer_model"):
    globals().pop(name, None)
free_gpu()
print("VRAM trước khi train:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

for setup in ["E1", "E2", "E3"]:
    if (Path("work/adapters") / setup / "run.json").exists():
        print(f"=== {setup}: đã có run.json, bỏ qua ===")
        continue
    print(f"=== huấn luyện {setup} ===")
    train_with_fallback(setup, splits["train"])
    print("VRAM sau khi dọn:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
    backup(setup)

In [ ]:

# Khôi phục model Qwen3-VL-4B-Instruct
import gc
import torch

free_gpu()

print("Đang tải model, vui lòng chờ...")
model = fresh_base_model()

print("\nModel đã tải:", type(model).__name__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM đang dùng:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


In [ ]:

import json
from pathlib import Path

# Đọc lại dữ liệu nếu biến splits đã mất
if "splits" not in globals():
    def read_jsonl(path):
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]

    splits = {
        s: read_jsonl(f"data/splits/{s}.jsonl")
        for s in ("train", "val", "test")
    }

train_entries = splits["train"]

print("Khôi phục dữ liệu thành công!")
print("Train:", len(splits["train"]), "ảnh")
print("Validation:", len(splits["val"]), "ảnh")
print("Test:", len(splits["test"]), "ảnh")
print("train_entries:", len(train_entries))

# Kiểm tra một mẫu dữ liệu
print("\nMẫu đầu tiên:")
print(train_entries[0].keys())
print("Image path:", train_entries[0]["image_path"])


In [ ]:

import gc
import torch
from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration

# Giải phóng model đang bị lượng tử hóa sai
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Khai báo rõ các lớp Linear của projector cần giữ nguyên
skip_modules = [
    "lm_head",
    "model.visual.merger.linear_fc1",
    "model.visual.merger.linear_fc2",
    "model.visual.deepstack_merger_list.0.linear_fc1",
    "model.visual.deepstack_merger_list.0.linear_fc2",
    "model.visual.deepstack_merger_list.1.linear_fc1",
    "model.visual.deepstack_merger_list.1.linear_fc2",
    "model.visual.deepstack_merger_list.2.linear_fc1",
    "model.visual.deepstack_merger_list.2.linear_fc2",
]

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=skip_modules,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map={"": 0},
)

print("Model đã nạp lại.")

# Kiểm tra các lớp quan trọng
from bitsandbytes.nn import Linear4bit

check_names = [
    "model.visual.merger.linear_fc1",
    "model.visual.merger.linear_fc2",
    "model.visual.deepstack_merger_list.0.linear_fc1",
    "model.visual.deepstack_merger_list.0.linear_fc2",
    "model.language_model.layers.0.self_attn.q_proj",
]

for name in check_names:
    module = model.get_submodule(name)
    print(
        name,
        "|", type(module).__name__,
        "|", module.weight.dtype,
        "|", tuple(module.weight.shape),
    )

print("\nProjector đã bỏ qua lượng tử hóa:",
      not isinstance(
          model.get_submodule("model.visual.merger.linear_fc1"),
          Linear4bit
      ))


In [ ]:
# Kiểm tra projector trước khi áp dụng E2
for name, p in model.named_parameters():
    if any(k in name for k in PROJ_KEYS):
        print(name, p.dtype, tuple(p.shape))
        if "linear_fc1.weight" in name or "linear_fc2.weight" in name:
            assert p.dtype == torch.float16, (
                f"Projector chưa ở FP16: {name} = {p.dtype}"
            )

print("Kiểm tra projector hoàn tất.")

In [ ]:

import re
import torch
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup, r=16, lora_alpha=None):
    """E1: LoRA LLM. E2: LoRA LLM + projector FP32. E3: thêm LoRA ViT."""
    assert setup in ("E1", "E2", "E3")
    lora_alpha = lora_alpha or 2 * r

    for p in model.parameters():
        p.requires_grad = False

    # Chuyển projector sang FP32 TRƯỚC khi bọc LoRA.
    if setup in ("E2", "E3"):
        for name, module in model.named_modules():
            if name.endswith(PROJ_KEYS):
                module.float()

    model.enable_input_require_grads()

    pattern = (
        LLM_PATTERN
        if setup != "E3"
        else f"(?:{LLM_PATTERN})|(?:{VIT_PATTERN})"
    )
    matched = [
        name for name, _ in model.named_modules()
        if re.fullmatch(pattern, name)
    ]

    n_llm = model.config.text_config.num_hidden_layers * 7
    n_vit = model.config.vision_config.depth * 4
    expected = n_llm if setup in ("E1", "E2") else n_llm + n_vit

    print(f"{setup}: {len(matched)} module LoRA; kỳ vọng {expected}")
    assert len(matched) == expected, "Số module LoRA không khớp"

    peft_model = get_peft_model(
        model,
        LoraConfig(
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=0.05,
            target_modules=matched,
            task_type="CAUSAL_LM",
        ),
    )

    if setup in ("E2", "E3"):
        for name, p in peft_model.named_parameters():
            if any(key in name for key in PROJ_KEYS):
                p.requires_grad = True

    trainable = [
        (name, p) for name, p in peft_model.named_parameters()
        if p.requires_grad
    ]
    print("Tổng tham số trainable:", f"{sum(p.numel() for _, p in trainable):,}")
    print("Dtype trainable:", {str(p.dtype) for _, p in trainable})

    if setup in ("E2", "E3"):
        bad = [
            (name, str(p.dtype))
            for name, p in trainable
            if any(key in name for key in PROJ_KEYS)
            and p.dtype != torch.float32
        ]
        assert not bad, f"Projector chưa ở FP32: {bad[:5]}"

    return peft_model, matched


In [ ]:
print("Model hiện tại:", type(model).__name__)
print("Projector merger dtype:", model.get_submodule(
    "model.visual.merger.linear_fc1"
).weight.dtype)
print("LLM q_proj:", type(model.get_submodule(
    "model.language_model.layers.0.self_attn.q_proj"
)).__name__)

In [ ]:
import gc
import torch

del peft_model
del matched
del model

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

model = fresh_base_model()

print("Đã nạp lại model nền sạch.")
print(
    "Projector:",
    model.get_submodule("model.visual.merger.linear_fc1").weight.dtype
)
print(
    "LLM:",
    type(model.get_submodule(
        "model.language_model.layers.0.self_attn.q_proj"
    )).__name__
)

In [ ]:
def fresh_base_model():
    return Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        dtype=torch.float16,
        device_map={"": 0},
    )

In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()

model = fresh_base_model()

for name in [
    "model.visual.merger.linear_fc1",
    "model.visual.deepstack_merger_list.0.linear_fc1",
    "model.language_model.layers.0.self_attn.q_proj",
]:
    module = model.get_submodule(name)
    print(name, type(module).__name__, module.weight.dtype)

In [ ]:
import inspect

src = inspect.getsource(train_setup)

for i, line in enumerate(src.splitlines(), 1):
    if any(k in line for k in [
        "GradScaler", "autocast", "scaler.scale",
        "scaler.unscale_", "scaler.step",
        "scaler.update", "backward"
    ]):
        print(f"{i:03}: {line}")

In [ ]:

def train_setup(model, processor, setup, train_entries, max_steps=300, lr=2e-4,
                batch_size=2, grad_accum=4, save_every=100,
                save_dir="work/adapters", r=16, grad_ckpt=False):

    peft_model, _ = apply_setup(model, setup, r=r)
    out_dir = Path(save_dir) / setup
    out_dir.mkdir(parents=True, exist_ok=True)

    trainable_params = [
        p for p in peft_model.parameters() if p.requires_grad
    ]
    print(
        f"{setup}: dtype tham số trainable =",
        {str(p.dtype) for p in trainable_params}
    )
    assert all(p.dtype == torch.float32 for p in trainable_params), (
        "Có tham số trainable không phải FP32"
    )

    opt = torch.optim.AdamW(
        trainable_params, lr=lr, weight_decay=0.0
    )
    sched = get_cosine_schedule_with_warmup(
        opt, max(1, int(0.05 * max_steps)), max_steps
    )

    if grad_ckpt:
        peft_model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )

    peft_model.config.use_cache = False
    peft_model.train()
    print(f"{setup}: gradient checkpointing = {grad_ckpt}")

    pool = ThreadPoolExecutor(max_workers=1)
    queue = deque(
        pool.submit(sample_batch, train_entries, batch_size, processor)
        for _ in range(2)
    )

    def save_checkpoint(step):
        ckpt_dir = out_dir / f"step{step}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        peft_model.save_pretrained(str(ckpt_dir))

        if setup in ("E2", "E3"):
            extra = {
                n: p.detach().cpu()
                for n, p in peft_model.named_parameters()
                if any(k in n for k in PROJ_KEYS) and p.requires_grad
            }
            torch.save(extra, ckpt_dir / "extra_trainable.pt")

        print(f"Đã lưu checkpoint: {ckpt_dir}")

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    data_wait = 0.0
    last_loss = None

    try:
        for step in range(1, max_steps + 1):
            opt.zero_grad(set_to_none=True)

            for _ in range(grad_accum):
                fut = queue.popleft()
                queue.append(
                    pool.submit(sample_batch, train_entries, batch_size, processor)
                )

                tw = time.time()
                cpu_batch = fut.result()
                data_wait += time.time() - tw

                batch = {
                    k: v.to(peft_model.device)
                    for k, v in cpu_batch.items()
                }

                with autocast(device_type="cuda", dtype=torch.float16):
                    loss = peft_model(**batch).loss

                if not torch.isfinite(loss):
                    raise RuntimeError(
                        f"Loss không hữu hạn ở bước {step}: {loss.item()}"
                    )

                (loss / grad_accum).backward()
                last_loss = loss.detach().item()

                del loss, batch, cpu_batch

            grad_norm = torch.nn.utils.clip_grad_norm_(
                trainable_params, 1.0
            )

            if not torch.isfinite(grad_norm):
                raise RuntimeError(
                    f"Gradient norm không hữu hạn tại bước {step}"
                )

            opt.step()
            sched.step()

            if step % 20 == 0 or step == max_steps:
                elapsed = time.time() - t0
                print(
                    f"[{setup}] bước {step}/{max_steps} "
                    f"loss={last_loss:.4f} "
                    f"{elapsed / step:.1f}s/bước "
                    f"chờ dữ liệu={data_wait / 60:.2f} phút"
                )

            if step % save_every == 0 or step == max_steps:
                save_checkpoint(step)

    finally:
        pool.shutdown(wait=True, cancel_futures=True)

    run_info = {
        "setup": setup,
        "rank": r,
        "trainable_params": sum(p.numel() for p in trainable_params),
        "peak_vram_gb": round(
            torch.cuda.max_memory_allocated() / 1e9, 2
        ),
        "train_minutes": round((time.time() - t0) / 60, 2),
        "final_step": step,
        "grad_checkpointing": grad_ckpt,
        "data_wait_minutes": round(data_wait / 60, 2),
        "n_train_images": len(train_entries),
    }

    (out_dir / "run.json").write_text(
        json.dumps(run_info, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(run_info)

    return peft_model, run_info


In [ ]:

free_gpu()
model = fresh_base_model()

# Xác nhận projector không bị lượng tử hóa
assert model.get_submodule(
    "model.visual.merger.linear_fc1"
).weight.dtype == torch.float16

assert model.get_submodule(
    "model.language_model.layers.0.self_attn.q_proj"
).__class__.__name__ == "Linear4bit"

peft_model, e2_info = train_setup(
    model=model,
    processor=processor,
    setup="E2",
    train_entries=train_entries,
    max_steps=300,
    lr=2e-4,
    batch_size=1,
    grad_accum=8,
    save_every=100,
    save_dir="work/adapters",
    r=16,
    grad_ckpt=True,
)


In [ ]:

from pathlib import Path
from scripts.labcommon import read_jsonl

test_entries = read_jsonl("data/splits/test.jsonl")
pred_path = Path("work/predictions/E2_test.jsonl")
pred_path.parent.mkdir(parents=True, exist_ok=True)

print("Số ảnh test:", len(test_entries))
print("Model:", type(eval_e2).__name__)
print("Processor:", type(processor).__name__)
print("File output:", pred_path)

run_inference(
    model=eval_e2,
    processor=processor,
    entries=test_entries,
    out_path=str(pred_path),
    batch_size=1,
)

print("File dự đoán tồn tại:", pred_path.exists())
if pred_path.exists():
    print("Dung lượng:", round(pred_path.stat().st_size / (1024**2), 2), "MB")


In [ ]:

import json
from pathlib import Path
from PIL import Image
import torch

pred_path = Path("work/predictions/E2_test.jsonl")

# Đọc các kết quả đã lưu, không xóa file
with pred_path.open("r", encoding="utf-8") as f:
    saved_lines = [line for line in f if line.strip()]

saved = {}
for line in saved_lines:
    item = json.loads(line)
    saved[item["image_id"]] = item

# Chỉ xử lý ảnh chưa có kết quả
remaining = [
    e for e in test_entries
    if e["image_id"] not in saved
]

print("Ảnh trong test:", len(test_entries))
print("Ảnh đã có kết quả:", len(saved))
print("Ảnh cần chạy tiếp:", len(remaining))

assert len(saved) == 240, (
    f"Số kết quả hiện tại là {len(saved)}, không phải 240. "
    "Dừng để kiểm tra, không ghi thêm."
)

if remaining:
    eval_e2.eval()

    with pred_path.open("a", encoding="utf-8") as f:
        for i, e in enumerate(remaining, start=1):
            try:
                img_path = Path("data") / e["image_path"]
                with Image.open(img_path) as im:
                    img = im.convert("RGB")

                messages = [{
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": PROMPT},
                    ],
                }]

                inputs = processor.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt",
                ).to(eval_e2.device)

                prompt_len = inputs["input_ids"].shape[1]

                with torch.inference_mode():
                    out = eval_e2.generate(
                        **inputs,
                        max_new_tokens=192,
                        do_sample=False,
                    )

                raw = processor.batch_decode(
                    out[:, prompt_len:],
                    skip_special_tokens=True,
                )[0]

                f.write(json.dumps({
                    "image_id": e["image_id"],
                    "raw": raw,
                }, ensure_ascii=False) + "\n")

                # Buộc ghi tiến độ sau mỗi ảnh
                f.flush()

            except Exception as exc:
                print(f"\nLỗi ở ảnh {e['image_id']}: {exc}")
                raise

            if i % 5 == 0 or i == len(remaining):
                print(
                    f"Đã xử lý tiếp {i}/{len(remaining)} ảnh; "
                    f"tổng kết quả: {len(saved) + i}/"
                    f"{len(test_entries)}"
                )

print("Hoàn tất cell tiếp tục inference.")


In [ ]:
import subprocess
import sys
from pathlib import Path

pred_path = Path("work/predictions/E2_test.jsonl")
result_path = Path("work/results/E2_test.json")

Path("work/results").mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/score.py",
    "--pred", str(pred_path),
    "--split", "test",
    "--coord", "norm1000",
    "--name", "E2",
    "--out", str(result_path),
    "--meta", "trainable_params=142135296",
    "--meta", "train_minutes=56.26",
    "--meta", "peak_vram_gb=8.47",
]

subprocess.run(cmd, check=True)

print("\nKết quả lưu tại:", result_path)

In [ ]:

# Giải phóng model đánh giá E2 nếu biến vẫn còn tồn tại
import gc
import torch

if "eval_e2" in globals():
    del eval_e2

if "eval_base" in globals():
    del eval_base

gc.collect()
torch.cuda.empty_cache()

# Tạo model nền mới
e3_base = fresh_base_model()

# Áp dụng cấu hình E3
e3_model, e3_matched = apply_setup(e3_base, "E3", r=16)

# Kiểm tra bộ nhớ và số tham số
trainable_params = sum(
    p.numel() for p in e3_model.parameters() if p.requires_grad
)

print("Số module LoRA:", len(e3_matched))
print(f"Số tham số trainable: {trainable_params:,}")
print(f"VRAM đang dùng: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM đã cấp phát: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


In [ ]:

import gc
import torch

# 1. Đảm bảo dữ liệu train đã được nạp
if "train_entries" not in globals():
    train_entries = read_jsonl("data/splits/train.jsonl")

assert len(train_entries) == 1422, (
    f"Số ảnh train không như dự kiến: {len(train_entries)}"
)

# 2. Giải phóng model cũ để giảm nguy cơ hết VRAM
for name in [
    "e3_model", "e3_base",
    "eval_e2", "eval_base",
    "model", "peft_model",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(f"Ảnh train: {len(train_entries)}")
print(f"VRAM trước khi train: "
      f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# 3. Train E3
e3_base = fresh_base_model()

e3_model, e3_info = train_setup(
    model=e3_base,
    processor=processor,
    setup="E3",
    train_entries=train_entries,
    max_steps=300,
    lr=2e-4,
    batch_size=2,
    grad_accum=4,
    save_every=100,
    save_dir="work/adapters",
    r=16,
    grad_ckpt=True,
)

print("\nKẾT QUẢ TRAIN E3")
print(e3_info)


In [ ]:

import gc
import json
import shutil
from pathlib import Path

import torch
import bitsandbytes as bnb
from google.colab import drive

# =========================================================
# 1. GIAI PHONG BO NHO
# =========================================================
gc.collect()
torch.cuda.empty_cache()

print("VRAM dang su dung:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("VRAM PyTorch da dat:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

# =========================================================
# 2. MOUNT GOOGLE DRIVE
# =========================================================
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ROS_VLM_Checkpoints/E3_safe_run"
)
LOCAL_ROOT = Path("work/adapters_e3_safe/E3")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# =========================================================
# 3. CAU HINH E3
# =========================================================
MAX_STEPS = 300
BATCH_SIZE = 1
GRAD_ACCUM = 8
SAVE_EVERY = 50
LR = 0.0002

# =========================================================
# 4. HAM LUU CHECKPOINT LOCAL + GOOGLE DRIVE
# =========================================================
def save_and_backup(model, optimizer, scheduler, step, losses):
    step_name = f"step{step:03d}"
    local_dir = LOCAL_ROOT / step_name

    local_dir.mkdir(parents=True, exist_ok=True)

    # Luu LoRA adapter
    model.save_pretrained(local_dir)

    # Luu cac projector dang trainable
    projector_state = {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
        and any(key in name for key in PROJ_KEYS)
    }

    if not projector_state:
        raise RuntimeError(
            "Khong tim thay projector trainable. "
            "Dung lai de tranh luu checkpoint khong day du."
        )

    torch.save(
        projector_state,
        local_dir / "extra_trainable.pt"
    )

    # Luu trang thai optimizer/scheduler de ho tro khoi phuc
    torch.save(
        {
            "step": step,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "losses": losses,
        },
        local_dir / "training_state.pt"
    )

    metadata = {
        "model_id": MODEL_ID,
        "experiment": "E3",
        "step": step,
        "max_steps": MAX_STEPS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "learning_rate": LR,
        "projector_tensor_count": len(projector_state),
        "latest_loss": losses[-1] if losses else None,
    }

    (local_dir / "run.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8"
    )

    # Copy vao thu muc tam tren Drive truoc
    drive_tmp = DRIVE_ROOT / f"{step_name}_uploading"
    drive_final = DRIVE_ROOT / step_name

    if drive_tmp.exists():
        shutil.rmtree(drive_tmp)

    shutil.copytree(local_dir, drive_tmp)

    # Kiem tra cac file quan trong truoc khi danh dau da luu
    required = [
        drive_tmp / "adapter_config.json",
        drive_tmp / "adapter_model.safetensors",
        drive_tmp / "extra_trainable.pt",
        drive_tmp / "training_state.pt",
        drive_tmp / "run.json",
    ]

    missing = [
        str(p.name) for p in required
        if not p.exists() or p.stat().st_size == 0
    ]

    if missing:
        raise RuntimeError(
            f"Checkpoint Drive chua day du: {missing}"
        )

    if drive_final.exists():
        shutil.rmtree(drive_final)

    drive_tmp.rename(drive_final)

    print(f"\nDa sao luu checkpoint: {drive_final}")
    print("Projector tensors:", len(projector_state))
    print("Kich thuoc file:")
    for p in sorted(drive_final.iterdir()):
        print(f"  {p.name}: {p.stat().st_size / 1024**2:.1f} MB")


# =========================================================
# 5. TRAIN E3
# =========================================================
def train_e3_safe(base_model, processor, train_entries):
    model, info = apply_setup(base_model, "E3", r=16)

    # Bat gradient checkpointing de giam VRAM
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False

    trainable = [p for p in model.parameters() if p.requires_grad]

    print("Trainable parameters:", sum(p.numel() for p in trainable))
    print("VRAM sau khi setup:",
          round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

    optimizer = bnb.optim.PagedAdamW8bit(
        trainable,
        lr=LR,
        weight_decay=0.01,
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=15,
        num_training_steps=MAX_STEPS,
    )

    losses = []
    optimizer.zero_grad(set_to_none=True)

    model.train()

    for step in range(1, MAX_STEPS + 1):
        accumulated_loss = 0.0

        for micro_step in range(GRAD_ACCUM):
            batch = sample_batch(
                train_entries,
                BATCH_SIZE,
                processor,
            )

            batch = {
                key: (
                    value.to("cuda")
                    if isinstance(value, torch.Tensor)
                    else value
                )
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                output = model(**batch)
                loss = output.loss / GRAD_ACCUM

            loss.backward()
            accumulated_loss += loss.detach().item()

            del batch, output, loss

        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        losses.append(accumulated_loss)

        if step == 1 or step % 10 == 0:
            print(
                f"Step {step}/{MAX_STEPS} | "
                f"loss={accumulated_loss:.4f} | "
                f"VRAM={torch.cuda.memory_allocated()/1024**3:.2f} GB"
            )

        # Sao luu moi 50 optimizer steps
        if step % SAVE_EVERY == 0:
            save_and_backup(
                model, optimizer, scheduler, step, losses
            )

    # Luu them checkpoint cuoi neu khong chia het cho SAVE_EVERY
    if MAX_STEPS % SAVE_EVERY != 0:
        save_and_backup(
            model, optimizer, scheduler, MAX_STEPS, losses
        )

    model.config.use_cache = True
    print("E3 da hoan thanh.")
    return model, info


# =========================================================
# 6. TAO MODEL MOI VA CHAY
# =========================================================
gc.collect()
torch.cuda.empty_cache()

e3_base = fresh_base_model()

e3_model, e3_info = train_e3_safe(
    e3_base,
    processor,
    train_entries,
)


## 7. Suy luận và chấm E1–E3

In [ ]:
from peft import PeftModel


def load_for_inference(setup, step=300, save_dir="work/adapters"):
    """model gốc 4-bit → adapter LoRA → nạp lại projector (nếu có)."""
    base = fresh_base_model()
    adapter_dir = Path(save_dir) / setup / f"step{step}"
    infer_model = PeftModel.from_pretrained(base, str(adapter_dir))

    extra_path = adapter_dir / "extra_trainable.pt"
    if extra_path.exists():
        extra = torch.load(extra_path, map_location="cpu")
        result = infer_model.load_state_dict(extra, strict=False)
        unexpected = [k for k in result.unexpected_keys if k in extra]
        assert not unexpected, f"key projector không khớp tên trong model: {unexpected[:3]}"
        print(f"  đã nạp {len(extra)} tensor projector từ {extra_path.name}")
    infer_model.eval()
    return infer_model


def infer_and_score(setup, save_dir, pred_path, name, out_json):
    """Suy luận test bằng adapter của `setup` trong `save_dir`, rồi chấm bằng scripts/score.py."""
    ri = json.loads((Path(save_dir) / setup / "run.json").read_text(encoding="utf-8"))
    m = load_for_inference(setup, step=ri["final_step"], save_dir=save_dir)
    t0 = time.time()
    run_inference(m, processor, splits["test"], pred_path)
    print(f"  suy luận {name}: {(time.time() - t0) / 60:.1f} phút")
    del m
    free_gpu()
    subprocess.run([
        "python", "scripts/score.py", "--pred", pred_path, "--name", name, "--out", str(out_json),
        "--meta", f"trainable_params={ri['trainable_params']}",
        "--meta", f"peak_vram_gb={ri['peak_vram_gb']}",
        "--meta", f"train_minutes={ri['train_minutes']}",
    ], check=True)


for name in ("model", "base", "peft_model", "infer_model"):
    globals().pop(name, None)
free_gpu()

FORCE = False   # True = suy luận + chấm lại cả những setup đã có kết quả
for setup in ["E1", "E2", "E3"]:
    evidence = Path(f"report/evidence/{setup.lower()}.json")
    if evidence.exists() and not FORCE:
        print(f"=== {setup}: đã có {evidence.name}, bỏ qua ===")
        continue
    print(f"=== suy luận + chấm {setup} ===")
    infer_and_score(setup, "work/adapters", f"work/{setup.lower()}.jsonl", setup, evidence)
    backup()

In [ ]:
# Kiểm tra: E1 phải hơn E0 ít nhất 0.10 F1 (rubric). Debug nhanh: dùng --split val.
for setup in ["E0", "E1", "E2", "E3"]:
    path = Path(f"report/evidence/{setup.lower()}.json")
    if path.exists():
        r = json.loads(path.read_text(encoding="utf-8"))
        m = r["micro"]
        print(f"{setup}: F1={m['f1']:.3f}  CI95={r['f1_ci95']}  meanIoU={r['mean_iou']:.3f}  "
              f"valid={r['parse_valid_rate']:.3f}  FP/img={r['fp_per_image']:.2f}")
for setup in ["E1", "E2", "E3"]:
    p = Path("work/adapters") / setup / "run.json"
    if p.exists():
        i = json.loads(p.read_text(encoding="utf-8"))
        print(f"{setup}: {i['train_minutes']} phút, VRAM đỉnh {i['peak_vram_gb']} GB, "
              f"chờ dữ liệu {i.get('data_wait_minutes')} phút, grad_ckpt={i.get('grad_checkpointing')}")

## 8. E4 (ablation): `data_size` = 10% vs 100%
Đổi đúng MỘT biến so với setup tốt nhất của E1–E3. Bản 100% chính là setup tốt nhất đã train nên chỉ chấm lại bản dự đoán đó
(không train thêm); chỉ phải train bản 10%. Xem `configs/e4.yaml` trong repo để chắc giá trị và tên file đúng.

In [ ]:
BEST_SETUP = "E2"   # ĐỔI thành setup có F1 cao nhất trong E1–E3 sau khi có kết quả
FRAC = 0.10

sub_n = max(1, int(len(splits["train"]) * FRAC))
subset = random.Random(0).sample(splits["train"], sub_n)
print(f"E4 data_size: {sub_n}/{len(splits['train'])} ảnh train ({FRAC:.0%})")

save_dir = "work/adapters_e4_data10"
out10 = Path("report/evidence/e4_data_size_10.json")
if out10.exists():
    print("E4 10%: đã có kết quả, bỏ qua")
else:
    for name in ("model", "base", "peft_model", "infer_model"):
        globals().pop(name, None)
    free_gpu()
    if not (Path(save_dir) / BEST_SETUP / "run.json").exists():
        train_with_fallback(BEST_SETUP, subset, save_dir=save_dir)   # cùng 300 bước, chỉ khác số ảnh
    infer_and_score(BEST_SETUP, save_dir, "work/e4_data10.jsonl", "E4_data_size_10", out10)
    backup(BEST_SETUP, save_dir)

# bản 100% = setup tốt nhất đã có: chấm lại đúng bản dự đoán cũ, không cần GPU
out100 = Path("report/evidence/e4_data_size_100.json")
if not out100.exists():
    ri = json.loads((Path("work/adapters") / BEST_SETUP / "run.json").read_text(encoding="utf-8"))
    subprocess.run([
        "python", "scripts/score.py", "--pred", f"work/{BEST_SETUP.lower()}.jsonl",
        "--name", "E4_data_size_100", "--out", str(out100),
        "--meta", f"trainable_params={ri['trainable_params']}",
        "--meta", f"peak_vram_gb={ri['peak_vram_gb']}",
        "--meta", f"train_minutes={ri['train_minutes']}",
    ], check=True)
backup()
for p in (out10, out100):
    r = json.loads(p.read_text(encoding="utf-8"))
    print(f"{p.stem}: F1={r['micro']['f1']:.3f}  CI95={r['f1_ci95']}")

## 9. Ảnh minh hoạ cho báo cáo (≥ 6 ảnh đúng/sai)

In [ ]:
from scripts.score import parse_output, iou_xyxy

BEST_PRED = f"work/{BEST_SETUP.lower()}.jsonl"
preds = {r["image_id"]: parse_output(r["raw"]) for r in read_jsonl(BEST_PRED)}


def match_stats(entry):
    gt = [{"label": o["label"], "bbox": tuple(yolo_to_box(o["label"], o["yolo"])["bbox_2d"])} for o in entry["objects"]]
    pr = preds.get(entry["image_id"]) or []
    used, tp = set(), 0
    for g in gt:
        best, bj = 0.0, None
        for j, p in enumerate(pr):
            if j in used or p["label"] != g["label"]:
                continue
            v = iou_xyxy(g["bbox"], p["bbox"])
            if v > best:
                best, bj = v, j
        if bj is not None and best >= 0.5:
            used.add(bj)
            tp += 1
    return gt, pr, tp


good, bad = [], []
for e in splits["test"]:
    if not e["objects"]:
        continue
    gt, pr, tp = match_stats(e)
    (good if tp == len(gt) == len(pr) else bad).append(e)
picks = good[:3] + bad[:3]

Path("report/figures").mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, e in zip(axes.flat, picks):
    gt, pr, tp = match_stats(e)
    img = Image.open(f"data/{e['image_path']}").convert("RGB")
    d = ImageDraw.Draw(img)
    W, H = img.size
    sc = lambda b: [b[0] / 1000 * W, b[1] / 1000 * H, b[2] / 1000 * W, b[3] / 1000 * H]
    for g in gt:
        d.rectangle(sc(g["bbox"]), outline="lime", width=3)
        d.text((g["bbox"][0] / 1000 * W, g["bbox"][1] / 1000 * H - 12), "GT " + g["label"], fill="lime")
    for p in pr:
        d.rectangle(sc(p["bbox"]), outline="red", width=2)
        d.text((p["bbox"][0] / 1000 * W, p["bbox"][3] / 1000 * H + 2), "PRED " + p["label"], fill="red")
    ax.imshow(img)
    ax.set_title(f"{'ĐÚNG' if e in good else 'SAI'}: {tp}/{len(gt)} GT khớp, {len(pr)} dự đoán", fontsize=10)
    ax.axis("off")
plt.suptitle(f"{BEST_SETUP}: xanh = nhãn đúng, đỏ = dự đoán")
plt.tight_layout()
plt.savefig("report/figures/predictions.png", dpi=120)
plt.show()

## 10. E5 (tuỳ chọn, +10 điểm): GRPO với reward IoU
Cell dưới chứa hàm `reward()` + test chạy được không cần GPU. Phần chạy GRPO tự viết theo GUIDE bước 9.

In [ ]:
from scripts.score import parse_output, iou_xyxy


def reward(completion, gt_boxes):
    """gt_boxes: list[{"label","bbox"}] thang 0-1000 (như _gt_boxes trong scripts/score.py)."""
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0

    pairs = sorted(
        ((iou_xyxy(g["bbox"], p["bbox"]), i, j)
         for i, g in enumerate(gt_boxes) for j, p in enumerate(pred)
         if g["label"] == p["label"]),
        reverse=True,
    )
    used_g, used_p, total_iou = set(), set(), 0.0
    for v, i, j in pairs:
        if v < 0.5 or i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        total_iou += v

    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * total_iou / denom


def _test_reward():
    gt = [{"label": "pothole", "bbox": (100, 100, 200, 200)}]
    assert reward('[{"label":"pothole","bbox_2d":[100,100,200,200]}]', gt) > 0.9
    assert reward("not json", gt) == 0.0
    assert reward('[{"label":"crack","bbox_2d":[100,100,200,200]}]', gt) < 0.3
    r_extra = reward('[{"label":"pothole","bbox_2d":[100,100,200,200]},'
                      '{"label":"pothole","bbox_2d":[500,500,600,600]}]', gt)
    assert r_extra < reward('[{"label":"pothole","bbox_2d":[100,100,200,200]}]', gt)
    assert reward("[]", []) == 1.0
    print("tất cả test reward() đều pass")


_test_reward()

## 11. Trước khi nộp
- `python scripts/check_submission.py` không còn FAIL.
- Commit theo bước (≥ 5 commit): notebook, `report/evidence/*.json`, `report/figures/`, `REPORT.md`.
- KHÔNG commit `data/`, `work/`, `.env`, `*.safetensors`, `*.bin`, `*.pt`. KHÔNG sửa `scripts/`, `tests/`.
- Trong REPORT ghi rõ: prompt tiếng Anh ngắn, `grad_checkpointing` (xem `run.json`), thời gian/VRAM thật của từng setup.